In [ ]:
#!pip install -q -U google-genai
import google.generativeai as genai
import os
import time
import json
import random
from tqdm import tqdm
import pandas as pd
import sqlite3

In [ ]:
API_KEY = os.getenv("GOOGLE_API_KEY")
MODEL_NAME = "gemini-2.5-pro-latest"
REQUESTS_PER_MINUTE = 5
SECONDS_TO_WAIT = 60 / REQUESTS_PER_MINUTE
BATCH_SIZE = 15
MAX_ITERATIONS = 10  
MIN_ITERATIONS = 3   
CONVERGENCE_THRESHOLD = 0.25 
THRESHOLD = 0
DB_PATH = "your_database_name.db"
TABLE_NAME = "reviews_table"

OUTPUT_FILE = "labeled_reviews_final.jsonl"
FAILED_BATCHES_FILE = "failed_batches.jsonl"

In [ ]:
PROMPT_TEMPLATE = """
You are an expert AI Product Analyst. Your task is to analyze a batch of customer reviews for a single product. For each review, you will score its usefulness, classify its type, and finally, provide an overall usefulness threshold for the batch.

**Task Definition:**
1.  **Score Usefulness:** Assign a `usefulness_score` on a **floating-point scale from 1.0 to 10.0**.
2.  **Classify Review Type:** Categorize each text as either a "Personal Review" or "Information".
    - **Personal Review:** The text expresses a subjective user experience, opinion, or feeling. It often uses "I", "my", "we".
    - **Information:** The text is objective and factual, like a news snippet, a feature list, or technical specifications. It lacks personal sentiment.
3.  **Determine Threshold:** After analyzing all reviews, provide a single `usefulness_threshold` score. Reviews scoring below this are generally not useful.

**Input Format:**
You will be provided a JSON array of review objects. Each object has a unique `id`, `source` and `review_text`.

**Output Format:**
Your response MUST be a single, valid JSON object with TWO top-level keys:
1.  `usefulness_threshold`: A single float number.
2.  `review_analysis`: An array of objects, where each object contains:
    - `id`: The original review ID.
    - `usefulness_score`: The calculated score (float).
    - `review_type`: The classification ("Personal Review" or "Information").

Do not include any other text, greetings, or explanations outside of this JSON structure.

**Analyze the following batch of reviews:**
{reviews_json}
"""

In [ ]:
def get_reviews_from_db(db_path, table_name):
    try:
        conn = sqlite3.connect(db_path)
        # Your table must have columns named 'id' and 'review_text'
        df = pd.read_sql_query(f"SELECT id, review_text FROM {table_name}", conn)
        conn.close()
        return df.to_dict('records')  # list of dict's containing each row
        
    except Exception as e:
        print(f"Error connecting to or reading from the database: {e}")
        return []

In [ ]:
def append_to_jsonl(data, filename):
    with open(filename, 'a', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')

In [ ]:
def main():
    if not API_KEY:
        print("Error: GOOGLE_API_KEY environment variable not set.")
        return
    iter_threshold = []
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel(MODEL_NAME)
    all_reviews = get_reviews_from_db(DB_PATH, TABLE_NAME)
    if not all_reviews:
        print("No reviews to process. Exiting.")
        return
    
    scores_data = {
        review['id']: {'scores': [], 'review_type':""}
        for review in all_reviews
    }

    for iteration in range(MAX_ITERATIONS):
        print(f"\n--- Starting Iteration {iteration + 1}/{MAX_ITERATIONS} ---")
        random.shuffle(all_reviews)
        review_batches = [all_reviews[i:i + BATCH_SIZE] for i in range(0, len(all_reviews), BATCH_SIZE)]
        progress_bar = tqdm(review_batches, desc=f"Iter {iteration + 1}")
        
        for batch in progress_bar:
            try:
                reviews_json_str = json.dumps(batch, indent=2)
                prompt = PROMPT_TEMPLATE.format(reviews_json=reviews_json_str)
                response = model.generate_content(prompt)
                cleaned_response = response.text.strip().replace('```json', '').replace('```', '').strip()
                result_json = json.loads(cleaned_response)
                labeled_data = result_json.get("review_analysis", [])
                iter_threshold[iteration] += result_json.get("usefulness_threshold")/len(review_batches)

                for item in labeled_data:
                    review_id = item['id']
                    if review_id in scores_data:
                        scores_data[review_id]['scores'].append(item['usefulness_score'])
                        scores_data[review_id]['review_type'] = item['review_type']

            except Exception as e:
                print(f"\nAn error occurred while processing a batch: {e}")
                append_to_jsonl([{"error": str(e), "batch_data": batch}], FAILED_BATCHES_FILE)
            
            finally:
                time.sleep(SECONDS_TO_WAIT)
        
        if iteration > 0:
            total_change = 0
            for review_id, data in scores_data.items():
                scores = data['scores']
                avg = sum(scores)/len(scores)
                total_change += abs(avg - data['scores'][-1])
                data['scores'] = [avg]
            avg_change = total_change / len(scores_data) if scores_data else 0
            THRESHOLD = min(iter_threshold) 

            print(f"Iteration {iteration + 1} complete. Average score change: {avg_change:.6f}")

            if iteration + 1 >= MIN_ITERATIONS or avg_change < CONVERGENCE_THRESHOLD:
                print(f"\nScores have converged after {iteration + 1} iterations. Stopping early.")
                break

    print("\nAll iterations complete. Saving final results...")

    final_results = []
    for review_id, data in scores_data.items():
        final_results.append({
            "id": review_id,
            "text":all_reviews[review_id],
            "usefullness_score": data['scores'],
            "review_type": data.get('review_type', 'N/A'), # Use .get for safety
        })
    
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for item in final_results:
            f.write(json.dumps(item) + '\n')
    print(f"Processing finished. Final stabilized labels saved to '{OUTPUT_FILE}'.")

if __name__ == "__main__":
    main()